In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
from sklearn.preprocessing import (MinMaxScaler, PowerTransformer, RobustScaler, StandardScaler)


sns.set_theme(style="whitegrid", palette="colorblind")


In [2]:
imoveis_sp = pd.read_csv(r"..\..\data\imoveis_sao_paulo.csv")

In [3]:
imoveis_sp.info()

<class 'pandas.DataFrame'>
RangeIndex: 29706 entries, 0 to 29705
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   bairro                29706 non-null  str    
 1   zona                  29706 non-null  str    
 2   tier                  29706 non-null  str    
 3   area                  29706 non-null  float64
 4   rooms                 29706 non-null  int64  
 5   parking spaces        29409 non-null  float64
 6   floor                 25252 non-null  float64
 7   total_floors          25252 non-null  float64
 8   transit_distance_min  29706 non-null  int64  
 9   furniture             29706 non-null  str    
 10  hoa (R$)              29201 non-null  float64
 11  rent amount (R$)      29706 non-null  float64
 12  property tax (R$)     29706 non-null  float64
 13  total (R$)            29706 non-null  float64
dtypes: float64(8), int64(2), str(4)
memory usage: 3.2 MB


In [4]:
imoveis_sp.head()

,bairro,zona,tier,area,rooms,parking spaces,floor,total_floors,transit_distance_min,furniture,hoa (R$),rent amount (R$),property tax (R$),total (R$)
0,Santo Amaro,Zona Sul,mid,119.0,3,0.0,11.0,15.0,3,not furnished,NaN,5050.0,590.25,6910.0
1,Vila Mariana,Zona Sul,premium,97.9,3,1.0,15.0,25.0,10,not furnished,1072.0,5060.0,833.17,6970.0
2,Ipiranga,Zona Sul,mid,45.5,1,0.0,1.0,15.0,27,furnished,520.0,2470.0,283.92,3280.0
3,Raposo Tavares,Zona Oeste,mid,180.4,4,1.0,8.0,15.0,94,not furnished,2274.0,6270.0,775.17,9320.0
4,Brooklin,Zona Sul,luxury,36.1,0,1.0,2.0,8.0,1,not furnished,570.0,2960.0,330.00,3860.0


In [5]:
nomes_colunas = {
    "tier" : "padrao",
    "rooms" : "quartos",
    "parking spaces" : "vagas_garagem",
    "floor" : "andar",
    "total_floors" : "andares_predio",
    "transit_distance_min" : "distancia_transporte",
    "furniture" : "mobiliado",
    "hoa (R$)" : "condominio",
    "rent amount (R$)" : "aluguel",
    "property tax (R$)" : "iptu",
    "total (R$)" : "custo_do_imovel",
}

In [8]:
imoveis = ( imoveis_sp
    .rename(columns = nomes_colunas)
    .drop_duplicates()
    .reset_index(drop=True)
    .assign(
       tipo_imovel = lambda df: np.where(
           df["andar"].fillna(0) == 0,
           "casa",
           "apartamento"
       ) 
    )
    .astype({
        "vagas_garagem": "Int64",
        "andar": "Int64",
        "andares_predio": "Int64"
    })
)

**Notas de estudo:**

- Raw Strings (r"") para manter a barra invertida no caminho original do arquivo.
- .assign() cria novas colunas ou modifica existentes, retornando uma nova cópia do DataFrame com as alterações (sem alterar o objeto original).
- lambda df: declara a função e define que ela receberá o DataFrame (referenciado localmente como df) como parâmetro de entrada.
- .fillna(0) == 0 Preenche temporariamente os valores nulos com 0 e faz a comparação.
- df["novo_nome"] no caso de usar já importada, df["novo_antigo"] quando importação e modificação são simultâneas.
- .astype() alterar os tipos de dados
- "Int64" que permite misturar números inteiros e valores nulos na mesma coluna pois "int" no Pandas não suporta valores nulos. 

In [9]:
imoveis.head()

,bairro,zona,padrao,area,quartos,vagas_garagem,andar,andares_predio,distancia_transporte,mobiliado,condominio,aluguel,iptu,custo_do_imovel,tipo_imovel
0,Santo Amaro,Zona Sul,mid,119.0,3,0,11,15,3,not furnished,NaN,5050.0,590.25,6910.0,apartamento
1,Vila Mariana,Zona Sul,premium,97.9,3,1,15,25,10,not furnished,1072.0,5060.0,833.17,6970.0,apartamento
2,Ipiranga,Zona Sul,mid,45.5,1,0,1,15,27,furnished,520.0,2470.0,283.92,3280.0,apartamento
3,Raposo Tavares,Zona Oeste,mid,180.4,4,1,8,15,94,not furnished,2274.0,6270.0,775.17,9320.0,apartamento
4,Brooklin,Zona Sul,luxury,36.1,0,1,2,8,1,not furnished,570.0,2960.0,330.00,3860.0,apartamento
